# First models: risk model, naive T-learner and placebo check

Treatment under study: aspirin (RXASP). Outcome: death or dependency at 6 months.
This notebook fits a first risk model, builds a naive T-learner and checks, with a
placebo treatment, how much of the estimated effect heterogeneity is noise.

## 1. Data

prepare_data drops the 150 patients with a missing outcome and returns:
- X: the 23 features plus RXHEP, the other randomised treatment;
- y: the outcome, 0/1;
- t: the treatment under study, used to split patients, not as a feature.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

from ml_med.modeling import make_preprocessor, prepare_data

FEATURES_PATH = Path("../data/processed/ist_features.parquet")

df = pd.read_parquet(FEATURES_PATH)
X, y, t = prepare_data(df, "RXASP")
X.shape, round(y.mean(), 3), round(t.mean(), 3)

((19285, 24), np.float64(0.629), np.float64(0.5))

## 2. Risk model

A classifier X → y with all patients, treated and untreated together. It is not
the causal model: it checks that the preprocessing pipeline works and gives a
reference for how well the outcome can be predicted from baseline features.
AUC is used because the model outputs probabilities and no threshold is needed.

In [2]:
def make_model():
    return Pipeline(
        [
            ("prep", make_preprocessor("RXASP")),
            ("clf", LogisticRegression(max_iter=1000)),
        ]
    )


scores = cross_val_score(make_model(), X, y, cv=5, scoring="roc_auc")
scores.round(3), round(scores.mean(), 3)

(array([0.797, 0.81 , 0.797, 0.785, 0.8  ]), np.float64(0.798))

AUC ≈ 0.80 in every fold: baseline features rank patients by risk well.

## 3. Naive T-learner

Two identical models: one fitted on treated patients, one on untreated.
For every patient, CATE = P(outcome | treated) − P(outcome | untreated),
each from its own model. Predictions here are in-sample.

In [3]:
treated = t.astype(bool)

model_1 = make_model().fit(X[treated], y[treated])
model_0 = make_model().fit(X[~treated], y[~treated])

cate = pd.Series(
    model_1.predict_proba(X)[:, 1] - model_0.predict_proba(X)[:, 1],
    index=X.index,
    name="cate",
)
cate.describe().round(4)

count    19285.0000
mean        -0.0123
std          0.0430
min         -0.2496
25%         -0.0368
50%         -0.0116
75%          0.0095
max          0.2313
Name: cate, dtype: float64

Mean −0.012: matches the trial's average effect (62.3% vs 63.5%).
Std 0.043, with tails at ±0.25: taken at face value, the effect would vary a lot
between patients. The next section checks whether this spread is real.

## 4. Placebo check

The treatment column is shuffled at random, so it has no effect on anyone:
the true CATE is zero for every patient. Any spread the T-learner produces
here is pure noise, and gives the scale of the noise in section 3.

In [4]:
rng = np.random.default_rng(0)
t_fake = pd.Series(rng.permutation(treated.values), index=t.index)

fake_1 = make_model().fit(X[t_fake], y[t_fake])
fake_0 = make_model().fit(X[~t_fake], y[~t_fake])

cate_fake = pd.Series(
    fake_1.predict_proba(X)[:, 1] - fake_0.predict_proba(X)[:, 1],
    index=X.index,
    name="cate_fake",
)
pd.concat([cate.describe(), cate_fake.describe()], axis=1).round(4)

,cate,cate_fake
count,19285.0000,19285.0000
mean,-0.0123,-0.0082
std,0.0430,0.0370
min,-0.2496,-0.1926
25%,-0.0368,-0.0297
50%,-0.0116,-0.0072
75%,0.0095,0.0137
max,0.2313,0.2284


With no true effect at all, the spread is 0.037 vs 0.043 on the real treatment,
and the tails are just as extreme. Almost all of the heterogeneity in section 3
is noise from subtracting two separately fitted models.

## 5. Conclusion

- The average effect is estimated correctly; individual CATEs from this naive
  T-learner are not credible and must not be shown to a user.
- The problem is the method, not the data: two models fitted on different
  halves drift apart by chance, and the difference amplifies in the tails.
- Before comparing estimators we need out-of-sample predictions (cross-fitting)
  and a metric that works without observing individual effects (uplift curve),
  with the placebo as a reference.